# Prithvi-100M — Wildfire Burn Scar Segmentation

**Environment**: Colab A100 (GPU required)

Fine-tunes Prithvi-EO-1.0-100M + FPN decoder on dNBR burn scar labels.
Trains 40 epochs. Best Val IoU: 0.3808 at epoch 20.

In [ ]:
# ── Instalación automática ────────────────────────────────────────────────────
# This cell checks if packages are ready.
# If not, installs them and restarts the kernel. Second Run All will work.
import subprocess, sys, os

needs_install = False
try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0), \
        f'numpy {np.__version__} < 2.0'
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK numpy={np.__version__} | terratorch ready — skipping install.')
except Exception as e:
    needs_install = True
    print(f'Installing dependencies ({e})...')

if needs_install:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy==2.0.2', '--force-reinstall'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'terratorch', 'einops', 'timm',
                    'segmentation-models-pytorch'], check=True)
    print('\n✓ Install complete. Restarting kernel...')
    os.kill(os.getpid(), 9)

In [ ]:
# ── Entorno ───────────────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

print(f'IN_COLAB = {IN_COLAB}')

In [ ]:
import os, random, subprocess, warnings
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'numpy  : {np.__version__}')
print(f'torch  : {torch.__version__}')
print(f'device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT_DIR = BASE / 'models';  CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS  = BASE / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    LOCAL = Path('/content/patches')
    LOCAL.mkdir(exist_ok=True)
    if not (LOCAL / 'images').exists():
        print('Copiando patches a /content/ (más rápido que Drive)...')
        subprocess.run(['cp', '-r', str(BASE / 'data/patches/images'),      str(LOCAL)], check=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/patches/masks_dnbr'),  str(LOCAL)], check=True)
        print('Copia completada.')
    else:
        print('Patches ya en /content/')
    IMG_DIR  = LOCAL / 'images'
    MASK_DIR = LOCAL / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/patches/images'
    MASK_DIR = BASE / 'data/patches/masks_dnbr'

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE      = 224
BATCH_SIZE    = 8
NUM_EPOCHS    = 20          # additional epochs (21 → 40)
START_EPOCH   = 21          # resume from here
BEST_IOU_PREV = 0.4198      # best IoU from previous session (epochs 1–20)
LR            = 1e-4        # warm restart from initial LR
VAL_FRAC      = 0.2
FIRE_WEIGHT   = 5.0
THRESHOLD     = 0.5
MODEL_NAME    = 'prithvi_burn_scar'

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths) and len(img_paths) > 0, \
    f'ERROR: {len(img_paths)} imgs, {len(mask_paths)} masks'
print(f'Patches: {len(img_paths)}')
print(f'MASK_DIR: {MASK_DIR}')
print(f'Reanudando desde época {START_EPOCH} | IoU previo: {BEST_IOU_PREV}')

In [ ]:
# ── Prithvi normalization (HLS pretraining stats) ────────────────────
# Bands in order: B02, B03, B04, B8A, B11, B12
PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449],
                         dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420],
                         dtype=np.float32)
# Indices in the 11-band stack: [B02,B03,B04,B08,B8A,B11,B12,NDVI,NBR,NDWI,MASK]
BAND_IDX = [0, 1, 2, 4, 5, 6]   # B02, B03, B04, B8A, B11, B12


class WildfireDataset(Dataset):
    """Dataset de cicatrices de incendio. Devuelve (6,224,224) y (1,224,224)."""

    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment
        self._t = (256 - IMG_SIZE) // 2   # offset para center-crop 256→224

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)   # (11,256,256)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)  # (256,256)

        img = img[BAND_IDX] / 10000.0     # (6,256,256) — DN → reflectancia
        img = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]

        t = self._t
        img  = img[:, t:t+IMG_SIZE, t:t+IMG_SIZE]    # (6,224,224)
        mask = mask[t:t+IMG_SIZE, t:t+IMG_SIZE]      # (224,224)

        if self.augment:
            if random.random() > 0.5: img, mask = np.flip(img, 2).copy(), np.flip(mask, 1).copy()
            if random.random() > 0.5: img, mask = np.flip(img, 1).copy(), np.flip(mask, 0).copy()
            if random.random() > 0.5:
                k = random.randint(1, 3)
                img  = np.rot90(img,  k, (1, 2)).copy()
                mask = np.rot90(mask, k, (0, 1)).copy()

        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)


print('WildfireDataset defined.')

In [ ]:
# ── Architecture ──────────────────────────────────────────────────────────────

class PrithviNeck(nn.Module):
    """Tokens (B,197,D) → feature map (B,D,14,14). Quita CLS y hace reshape."""
    def __init__(self, n_side=14):
        super().__init__()
        self.n_side = n_side

    def forward(self, layers_out):
        x = layers_out[-1][:, 1:, :]           # remove CLS → (B,196,D)
        B, L, D = x.shape
        return x.permute(0,2,1).reshape(B, D, self.n_side, self.n_side)


class FPNDecoder(nn.Module):
    """Upsampling 14→224 con ConvTranspose2d. BN+GELU evita checkerboard."""
    def __init__(self, in_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModel(nn.Module):
    def __init__(self, backbone, embed_dim=768, n_side=14):
        super().__init__()
        self.backbone = backbone
        self.neck     = PrithviNeck(n_side)
        self.decoder  = FPNDecoder(embed_dim)

    def forward(self, x):               # x: (B,6,224,224)
        feats = self.backbone(x.unsqueeze(2))   # (B,6,1,224,224) → 12×(B,197,768)
        sp    = self.neck(feats)        # (B,768,14,14)
        return self.decoder(sp)         # (B,1,224,224)


print('Architecture defined.')

In [ ]:
# ── Load Prithvi (with EfficientNet-b4 fallback) ───────────────────────────
from terratorch.registry import BACKBONE_REGISTRY

USE_PRITHVI = False
model       = None

print('Loading prithvi_eo_v1_100 via terratorch...')
try:
    backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100').to(DEVICE)
    backbone.eval()

    # Test forward pass
    with torch.no_grad():
        _x   = torch.zeros(1, 6, 224, 224, device=DEVICE).unsqueeze(2)
        _out = backbone(_x)
    OUT_DIM = _out[-1].shape[-1]               # 768
    N_SIDE  = int((_out[-1].shape[1] - 1)**0.5)  # 14
    del _x, _out
    torch.cuda.empty_cache()

    # Congelar encoder
    for p in backbone.parameters():
        p.requires_grad = False

    model = PrithviSegModel(backbone, embed_dim=OUT_DIM, n_side=N_SIDE).to(DEVICE)
    USE_PRITHVI = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'\n✓ Prithvi ready | D={OUT_DIM} | grid={N_SIDE}×{N_SIDE}')
    print(f'  {n_total/1e6:.1f}M total | {n_train/1e6:.2f}M trainable (neck+decoder)')

except Exception as e:
    import traceback; traceback.print_exc()
    print(f'\n✗ Prithvi failed: {e}')
    print('→ Using EfficientNet-b4 as fallback')
    model = smp.Unet(
        encoder_name='efficientnet-b4', encoder_weights='imagenet',
        in_channels=6, classes=1, activation=None
    ).to(DEVICE)
    MODEL_NAME = 'efficientnet_b4'

# Verificar forward pass del modelo completo
with torch.no_grad():
    _o = model(torch.zeros(2, 6, IMG_SIZE, IMG_SIZE).to(DEVICE))
assert _o.shape == (2, 1, IMG_SIZE, IMG_SIZE), f'Unexpected forward shape: {_o.shape}'
print(f'Forward OK → {_o.shape}')
print(f'USE_PRITHVI = {USE_PRITHVI}  |  MODEL_NAME = {MODEL_NAME}')

In [ ]:
# ── Split + DataLoaders ────────────────────────────────────────────────────────
print('Scanning fire flags...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0 for p in tqdm(mask_paths)
], dtype=np.int32)

indices = np.arange(len(img_paths))
train_idx, val_idx = train_test_split(
    indices, test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED
)

print(f'Train: {len(train_idx)} ({fire_flags[train_idx].sum()} positive)')
print(f'Val  : {len(val_idx)}  ({fire_flags[val_idx].sum()} positive)')

train_fire = fire_flags[train_idx]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(np.where(train_fire==1, FIRE_WEIGHT, 1.0)),
    num_samples=len(train_idx), replacement=True
)

train_ds = WildfireDataset(
    [img_paths[i] for i in train_idx], [mask_paths[i] for i in train_idx], augment=True)
val_ds   = WildfireDataset(
    [img_paths[i] for i in val_idx],   [mask_paths[i] for i in val_idx],   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

In [ ]:
# ── Loss + Optimizer ──────────────────────────────────────────────────────────
dice_loss  = DiceLoss(mode='binary', from_logits=True)
focal_loss = FocalLoss(mode='binary', gamma=2.0, alpha=0.75)

def criterion(pred, target):
    return dice_loss(pred, target) + focal_loss(pred, target)

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

def fire_iou(logits, target, threshold=THRESHOLD):
    """IoU real para clase positiva. Devuelve None si no hay positivos en el batch."""
    pred = (logits.sigmoid() > threshold).float()
    if target.sum() == 0 and pred.sum() == 0:
        return None
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    return (tp / (tp + fp + fn + 1e-6)).item()

print(f'Optimizer: AdamW | LR={LR} | Trainable params: {sum(p.numel() for p in trainable):,}')

In [ ]:
# ── Load checkpoint and resume training ────────────────────────────────
ckpt_path = CKPT_DIR / f'best_{MODEL_NAME}_wildfire.pth'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
print(f'Checkpoint loaded: {ckpt_path.name}')
print(f'Previous IoU (epochs 1-20): {BEST_IOU_PREV:.4f} — resuming...\n')

best_val_iou = BEST_IOU_PREV
train_losses, val_losses, val_ious = [], [], []
END_EPOCH = START_EPOCH + NUM_EPOCHS - 1

for epoch in range(START_EPOCH, START_EPOCH + NUM_EPOCHS):

    # Train
    model.train()
    if USE_PRITHVI:
        model.backbone.eval()
    ep_loss = 0.0
    for imgs, masks in tqdm(train_loader,
                            desc=f'Epoch {epoch:02d}/{END_EPOCH} [train]', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward(); optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    train_losses.append(ep_loss / len(train_loader))

    # Val
    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader,
                                desc=f'Epoch {epoch:02d}/{END_EPOCH} [val]', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds  = model(imgs)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None:
                v_iou_list.append(iou)
    val_losses.append(v_loss / len(val_loader))
    val_ious.append(np.mean(v_iou_list) if v_iou_list else 0.0)

    print(f'Epoch {epoch:02d} | Train Loss: {train_losses[-1]:.4f} | '
          f'Val Loss: {val_losses[-1]:.4f} | Val IoU: {val_ious[-1]:.4f}')

    if val_ious[-1] > best_val_iou:
        best_val_iou = val_ious[-1]
        torch.save(model.state_dict(), ckpt_path)
        print(f'  → Saved (IoU: {best_val_iou:.4f})')

print(f'\nBest total Val IoU (epochs 1-{END_EPOCH}): {best_val_iou:.4f}')
print(f'Improvement vs previous session: {best_val_iou - BEST_IOU_PREV:+.4f}')

In [ ]:
# ── Training curves (epochs 21–40) ────────────────────────────────────
epochs = range(START_EPOCH, START_EPOCH + NUM_EPOCHS)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, label='Train'); axes[0].plot(epochs, val_losses, label='Val')
axes[0].set(xlabel='Epoch', ylabel='Loss', title=f'Loss — {MODEL_NAME} (resumed)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_ious, color='green', label='Val IoU')
axes[1].axhline(BEST_IOU_PREV, color='gray',  ls=':', label=f'Previous session: {BEST_IOU_PREV:.4f}')
axes[1].axhline(best_val_iou,  color='red',   ls='--', label=f'Best total: {best_val_iou:.4f}')
axes[1].set(xlabel='Epoch', ylabel='IoU', title='Val IoU — resumed')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / f'training_curves_{MODEL_NAME}_resumed.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Evaluación ────────────────────────────────────────────────────────────────
ckpt = CKPT_DIR / f'best_{MODEL_NAME}_wildfire.pth'
model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
model.eval()

all_probs, all_targets = [], []
patch_probs_list, patch_masks_list, patch_imgs_list, patch_ious = [], [], [], []

with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='Evaluating'):
        probs    = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
        masks_np = masks.squeeze(1).cpu().numpy()
        all_probs.append(probs.reshape(-1))
        all_targets.append(masks_np.reshape(-1))
        for b in range(len(imgs)):
            p = (probs[b] > THRESHOLD).astype(np.float32); m = masks_np[b]
            tp = (p*m).sum(); fp = (p*(1-m)).sum(); fn = ((1-p)*m).sum()
            patch_ious.append(float(tp/(tp+fp+fn+1e-6)))
            patch_probs_list.append(probs[b])
            patch_masks_list.append(masks_np[b])
            patch_imgs_list.append(imgs[b].numpy())

all_probs   = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets).astype(np.int32)
all_preds   = (all_probs > THRESHOLD).astype(np.int32)
patch_ious  = np.array(patch_ious)

prec, rec, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
tp = int(((all_preds==1)&(all_targets==1)).sum())
fp = int(((all_preds==1)&(all_targets==0)).sum())
fn = int(((all_preds==0)&(all_targets==1)).sum())
iou_pixel = tp / (tp + fp + fn + 1e-6)

fire_idx   = [i for i,f in enumerate(fire_flags[val_idx]) if f==1]
fire_ious  = patch_ious[fire_idx]

print('\n' + '='*55)
print(f'  RESULTS — {MODEL_NAME.upper()}')
print('='*55)
print(f'  Precision  : {prec:.4f}')
print(f'  Recall     : {rec:.4f}')
print(f'  F1-Score   : {f1:.4f}')
print(f'  IoU (pixel): {iou_pixel:.4f}')
print(f'  IoU (fire patches): {fire_ious.mean():.4f} ± {fire_ious.std():.4f}')
print('─'*55)
print('  COMPARISON:')
print(f'  FIRMS labels (real IoU): ~0.013   ← previous session')
print(f'  dNBR labels  (real IoU): {iou_pixel:.4f}  ← this model')
print('='*55)

# ── Save probabilities for local threshold sweep ─────────────────────────────
np.save(RESULTS / "val_probs.npy",   all_probs.astype(np.float32))
np.save(RESULTS / "val_labels.npy",  all_targets.astype(np.uint8))
print(f"Saved val_probs.npy + val_labels.npy to {RESULTS}")


In [ ]:
# ── Visualización — top 3 predicciones ───────────────────────────────────────
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

best_idx = sorted(fire_idx, key=lambda i: patch_ious[i], reverse=True)[:3]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for row, vi in enumerate(best_idx):
    img_np  = patch_imgs_list[vi]   # (6,224,224)
    prob_np = patch_probs_list[vi]
    mask_np = patch_masks_list[vi]
    pred_b  = (prob_np > THRESHOLD).astype(np.float32)

    def dn(ch):
        return norm_band(img_np[ch] * PRITHVI_STD[ch] + PRITHVI_MEAN[ch])
    rgb = np.dstack([dn(2), dn(1), dn(0)])

    err = np.zeros((*mask_np.shape, 3))
    err[(pred_b==1)&(mask_np==1)] = [0.0, 0.8, 0.0]
    err[(pred_b==1)&(mask_np==0)] = [1.0, 0.5, 0.0]
    err[(pred_b==0)&(mask_np==1)] = [0.9, 0.1, 0.1]

    axes[row,0].imshow(rgb);                                axes[row,0].axis('off')
    axes[row,1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1); axes[row,1].axis('off')
    axes[row,2].imshow(prob_np, cmap='hot',  vmin=0, vmax=1); axes[row,2].axis('off')
    axes[row,3].imshow(err);                                axes[row,3].axis('off')
    axes[row,0].set_ylabel(f'IoU={patch_ious[vi]:.3f}', fontsize=9)

for ax, t in zip(axes[0], ['RGB', 'Ground truth (dNBR)', 'Prediction (prob)', 'TP/FP/FN']):
    ax.set_title(t, fontsize=10)

tp_p = mpatches.Patch(color=[0,.8,0], label='TP'); fp_p = mpatches.Patch(color=[1,.5,0], label='FP')
fn_p = mpatches.Patch(color=[.9,.1,.1], label='FN')
fig.legend(handles=[tp_p,fp_p,fn_p], loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.01))
plt.suptitle(f'{MODEL_NAME} | Best Val IoU={best_val_iou:.4f}', fontsize=12, y=1.01)
plt.tight_layout()
out = RESULTS / f'predictions_{MODEL_NAME}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

---

## Threshold Optimization

Sweep thresholds 0.05 → 0.95 on the validation set to find the operating point
that maximizes F1 (or IoU). Requires  and  from the
evaluation cell above — no additional inference needed.

In [ ]:
# ── Threshold sweep — runs locally once val_probs.npy is saved ──────────────
# If running locally: load from Drive. If in Colab: use all_probs from cell above.
from pathlib import Path as _P
import numpy as _np

_probs_path  = RESULTS / "val_probs.npy"
_labels_path = RESULTS / "val_labels.npy"

if _probs_path.exists() and not "all_probs" in dir():
    all_probs   = _np.load(_probs_path).astype(np.float64)
    all_targets = _np.load(_labels_path).astype(np.int32)
    print(f"Loaded from Drive: {len(all_probs):,} pixels")
else:
    print(f"Using in-memory all_probs: {len(all_probs):,} pixels")

thresholds = np.arange(0.05, 0.96, 0.025)
results = []

for t in thresholds:
    preds_t = (all_probs > t).astype(np.int32)
    tp = int(((preds_t==1)&(all_targets==1)).sum())
    fp = int(((preds_t==1)&(all_targets==0)).sum())
    fn = int(((preds_t==0)&(all_targets==1)).sum())
    iou_t  = tp / (tp + fp + fn + 1e-6)
    prec_t = tp / (tp + fp + 1e-6)
    rec_t  = tp / (tp + fn + 1e-6)
    f1_t   = 2 * prec_t * rec_t / (prec_t + rec_t + 1e-6)
    results.append((t, iou_t, f1_t, prec_t, rec_t))

results      = np.array(results)
best_f1_idx  = results[:, 2].argmax()
best_iou_idx = results[:, 1].argmax()

print(f"Optimal threshold (F1)  : {results[best_f1_idx,  0]:.3f}  "
      f"-> IoU={results[best_f1_idx, 1]:.4f}  F1={results[best_f1_idx, 2]:.4f}  "
      f"Prec={results[best_f1_idx, 3]:.4f}  Rec={results[best_f1_idx, 4]:.4f}")
print(f"Optimal threshold (IoU) : {results[best_iou_idx, 0]:.3f}  "
      f"-> IoU={results[best_iou_idx,1]:.4f}  F1={results[best_iou_idx,2]:.4f}  "
      f"Prec={results[best_iou_idx,3]:.4f}  Rec={results[best_iou_idx,4]:.4f}")

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(results[:,0], results[:,1], label="IoU",       color="steelblue", lw=2)
axes[0].plot(results[:,0], results[:,2], label="F1",        color="green",     lw=2)
axes[0].plot(results[:,0], results[:,3], label="Precision", color="darkorange",lw=2, ls="--")
axes[0].plot(results[:,0], results[:,4], label="Recall",    color="crimson",   lw=2, ls="--")
axes[0].axvline(0.5, color="gray", ls=":", label="Baseline (0.50)")
axes[0].axvline(results[best_f1_idx, 0], color="green", ls=":",
                label=f"Best F1 (t={results[best_f1_idx,0]:.2f})")
axes[0].set(xlabel="Threshold", ylabel="Score", title="Metrics vs Threshold")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(results[:,4], results[:,3], color="steelblue", lw=2)
axes[1].scatter([results[best_f1_idx,4]], [results[best_f1_idx,3]],
                color="green", s=80, zorder=5, label=f"t={results[best_f1_idx,0]:.2f} (best F1)")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle("Threshold Optimization — Prithvi-100M + FPN", fontsize=11)
plt.tight_layout()
out = RESULTS / "threshold_sweep.png"
plt.savefig(out, dpi=150, bbox_inches="tight"); plt.show()
print(f"Saved: {out}")
